# **GEOGRAPHIC HISTORY PROBLEM SOLVING NOTEBOOK**
This notebook will compare how different datasets involved in the project handle changing administrative geographies to help determine the best strategies for creating a balanced panel and stable geographic units overtime. 

### SETUP

In [17]:
# Packages
import pandas as pd

# Files
laus = pd.read_csv('../../data_clean/unemployment_WIP.csv')
county_apps = pd.read_csv('../../data_clean/annual_county_applications.csv')
fips = pd.read_csv('../../data_intermediate/county_geo_id.csv')

### KNOWN ISSUES

* Connecticut is an issue across the board
* Four historic Alaska counties in LAUS not represented in FIPS
    * 'prince of wales-outer ketchikan census area', 'skagway-hoonah-angoon census area', 'valdez-cordova census area', 'wrangell-petersburg census area'

* Geographic change areas in County Apps  (below)



In [48]:
# list of geographic areas and all of their related areas from all missing CA counties:

geo_change_names = [
    'prince of wales-outer ketchikan census area',
    'prince of wales-hyder census area',
    'ketchikan gateway borough',
    'wrangell city and borough',

    'skagway-hoonah-angoon census area',
    'skagway municipality',
    'hoonah-angoon census area',

    'wrangell-petersburg census area',
    'petersburg census area',
    'petersburg borough',

    'valdez-cordova census area',
    'chugach census area',
    'copper river census area',

    'wade hampton census area',
    'kusilvak census area',

    'shannon county',
    'oglala lakota county',

    'bedford city',
    'bedford county'

    'kalawao county'
]

### TRACKING DOWN GEOGRAPHY CHANGES ACROSS DATASETS

Code below, findings here:
* Alaska excluded from preliminary analysis due to multiple changes in county-equivalent boundaries during the study period and differences in the treatment of historical geography across source datasets. Proper inclusion requires construction of geographically consistent units and variable-specific aggregation of outcomes. Will handle before next stage of analysis!
* Shannon County → Oglala Lakota County: harmonize/recode because it's a rename/recode rather than a substantive split
* Bedford city/Bedford County: Aggregate where excluded
* Kalawao county (not investigated here): aggregate with Maui county in datasets where split

In [49]:
# LAUS CHECK

# Keep only relevant historical geography names 
laus_geo = laus.loc[
    laus['county_name_match'].isin(geo_change_names)
].copy()

# Useful summary table
laus_geo_summary = (
    laus_geo
    .groupby('county_name_match')
    .agg(
        first_year=('year', 'min'),
        last_year=('year', 'max'),
        n_obs=('year', 'size'),
        n_years=('year', 'nunique')
    )
    .reset_index()
    .sort_values(['first_year', 'county_name_match'])
)

laus_geo_summary

,county_name_match,first_year,last_year,n_obs,n_years
3,ketchikan gateway borough,1990.0,2026.0,434,37
4,kusilvak census area,1990.0,2026.0,434,37
5,oglala lakota county,1990.0,2026.0,434,37
8,shannon county,1990.0,2026.0,434,37
2,hoonah-angoon census area,2010.0,2026.0,194,17
6,petersburg borough,2010.0,2026.0,194,17
7,prince of wales-hyder census area,2010.0,2026.0,194,17
9,skagway municipality,2010.0,2026.0,194,17
10,wrangell city and borough,2010.0,2026.0,194,17
0,chugach census area,2020.0,2026.0,74,7


In [50]:
# COUNTY APPLICATIONS CHECK 

# Creating standardized matching name
county_apps['county_name_match'] = county_apps['COUNTY_NAME'].str.lower().str.strip()

# Keep only relevant historical geography names
county_apps_geo = county_apps.loc[
    county_apps['county_name_match'].isin(geo_change_names)
].copy()

# Turning blank strings to NA
county_apps_geo['apps_numeric'] = pd.to_numeric(
    county_apps_geo['county_apps'],
    errors='coerce'
)

ca_with_data = county_apps_geo.loc[
    county_apps_geo['apps_numeric'].notna()
]

county_apps_geo_summary = (
    ca_with_data
    .groupby(['STATE', 'county_name_match'])
    .agg(
        first_year=('year', 'min'),
        last_year=('year', 'max'),
        n_years=('year', 'nunique'),
        n_obs=('year', 'size')
    )
    .reset_index()
)

county_apps_geo_summary



,STATE,county_name_match,first_year,last_year,n_years,n_obs
0,AK,chugach census area,2020,2024,5,5
1,AK,copper river census area,2020,2024,5,5
2,AK,hoonah-angoon census area,2005,2024,20,20
3,AK,ketchikan gateway borough,2005,2024,20,20
4,AK,kusilvak census area,2016,2024,9,9
5,AK,petersburg borough,2005,2024,20,20
6,AK,prince of wales-hyder census area,2005,2024,20,20
7,AK,skagway municipality,2005,2024,20,20
8,AK,valdez-cordova census area,2005,2019,15,15
9,AK,wade hampton census area,2005,2015,11,11


In [51]:
# FIPS CHECK

# Keep only relevant historical geography names 
fips_geo = fips.loc[
    fips['county_name_match'].isin(geo_change_names)
].copy()

fips_geo.to_csv('fips_geo_hist.csv')

### SHANNON / OGLALA LAKOTA COUNTY

Goal: investigate how to handle consolidating this area into one unit overtime across the different files

Outcome/Decision: leave LAUS as is, add shannon county data to Oglala Lakota County data to fill in empty years

In [56]:
oglala_names = ['shannon county', 'oglala county', 'oglala lakota county']
laus_oglala = laus.loc[laus['county_name_match'].isin(oglala_names)]
    # entirely complete Oglala Lakota County, no SD Shannon, no action needed
ca_oglala = county_apps.loc[county_apps['county_name_match'].isin(oglala_names)]
    # oglala lakota county empty until 2016, last year shannon data 2015
    # add shannon county data to Oglala Lakota County data to fill in empty years

print(ca_oglala.dtypes)

STATE                  str
COUNTY_NAME            str
full_fips            int64
state_fips           int64
county_fips          int64
year                 int64
county_apps            str
county_name_match      str
dtype: object


### BEDFORD CITY / BEDFORD COUNTY

Goal: Investigate which files need to have these areas consolidated 

In [57]:
laus_bedford = laus.loc[laus['county_name_match'].str.contains('bedford')]
laus_bedford = laus_bedford[laus_bedford['STATE'] == 'VA']
    # LAUS is entirely just bedford county, no city separation

ca_bedford = county_apps.loc[county_apps['county_name_match'].str.contains('bedford')]
ca_bedford = ca_bedford[ca_bedford['STATE'] == 'VA']
    # bedford city is a separate entity with data until 2014 (no data starting in 2014)
    # add data to bedford county and drop city
    

### KALAWAO COUNTY

Goal: see which datasets have separated Kalawao County

Decision: LAUS combines Kalawao, CA does not, so for all files with separate Kalawao County, combine with Maui county

In [58]:
laus_kalawao = laus[laus['county_name_match'].str.contains('kalawao')]
    # no kalawao to be found, must accounted within Maui county
ca_kalawao = county_apps[county_apps['county_name_match'].str.contains('kalawao')]
    # Kalawao is entirely separated throughout the period, needs to be combined to match